# Conservative charging: capacity and queue comparison

读取 `benchmark_conservative_charging.py` 保存的本地数据，对比 **current**（已有占用及确定在途车辆，再使用当前求解器的时间窗修复）和 **conservative**（额外假设所有潜在候选到达，保守筛选可立即开始的动作）。

实验为 **500 辆总车辆、单轮决策的合成 NYC API 场景**，包含正在充电、已确定在途及本轮决策车辆；默认 3/5 站、每站 50 slots、四个场景、10 个配对种子。实际配置和完成数以本次 `metadata.json` / `summary.csv` 为准。这里没有运行全天真实 NYC 轨迹。

- `available` 保留每辆决策车辆的等待动作，用于隔离充电候选筛选的影响；切换为 `nyc` 可读取使用 NYC 低电量等待限制的实验。
- 排队数据来自 **ChargingStation API 的固定日历回放，同一 epoch 先释放再到达**，不能直接证明完整 `NYC.step` 时序下零排队。
- 空置比例 = 未使用 slot-epochs /（站点数 × 每站 slots × 共同观察窗口 epochs）。包含背景占用；不把虚拟排队尾部拉长为观察窗口。它不是“可消除浪费”的证明。
- `assignment_score` 是合成、量化的固定分配分数，**不是美元 reward**。当前方法的外层时间窗贪心修复不作为全局区间调度最优基准。

运行所有单元格即可画图；仅在本次运行目录的 `figures/` 下保存 PNG、PDF，不改动原始 CSV、JSON 和 NPZ。


In [ ]:
from pathlib import Path
from itertools import product
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, MaxNLocator
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display

# 可手填绝对路径，或相对项目根目录的路径；None 自动找最新完整运行。
RUN_DIR = None
WAIT_POLICY = "available"  # 改为 "nyc" 读取原生低 SOC wait 规则的运行。
VEHICLES = 500
SCENARIO_ORDER = ["synchronized", "staggered", "committed", "mixed_demand"]
STATION_COUNTS = [3, 5]
METHODS = ["current", "conservative"]
COLORS = {"current": "#263F60", "conservative": "#2B9F91"}
METHOD_LABELS = {"current": "Current", "conservative": "Conservative"}
SCENARIO_LABELS = {"synchronized": "Synchronized", "staggered": "Staggered",
                   "committed": "Committed", "mixed_demand": "Mixed demand"}

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "benchmark_conservative_charging.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 icaps 项目目录启动 notebook，或将 PROJECT_ROOT 改为项目绝对路径。")
RESULT_ROOT = PROJECT_ROOT / "results" / "conservative_charging"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "axes.titlesize": 12, "axes.labelsize": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.axisbelow": True, "savefig.dpi": 200,
                     "pdf.fonttype": 42, "ps.fonttype": 42})


In [ ]:
def load_complete_run(path, wait_policy=WAIT_POLICY):
    """Complete means all cases requested in that run have rows, including failed solves."""
    path = Path(path)
    metadata = json.loads((path / "metadata.json").read_text())
    args = metadata["args"]
    if args["wait_policy"] != wait_policy or int(args["vehicles"]) != VEHICLES:
        raise ValueError(f"配置不匹配: wait_policy={args['wait_policy']}, vehicles={args['vehicles']}")
    summary = pd.read_csv(path / "summary.csv")
    raw = pd.read_csv(path / "raw.csv")
    expected = set(product(args["scenarios"], args["stations"], args["seeds"], METHODS))
    actual = set(raw[["scenario", "station_count", "seed", "method"]].itertuples(index=False, name=None))
    expected_summary = set(product(args["scenarios"], args["stations"], METHODS))
    actual_summary = set(summary[["scenario", "station_count", "method"]].itertuples(index=False, name=None))
    if actual != expected or len(raw) != len(expected):
        raise ValueError("raw.csv 尚未完整，或存在重复 case。")
    if actual_summary != expected_summary or len(summary) != len(expected_summary):
        raise ValueError("summary.csv 尚未完整，或存在重复 group。")
    if not summary.num_seeds.eq(len(args["seeds"])).all():
        raise ValueError("summary.csv 的种子数量与 metadata 不一致。")
    if not raw.wait_policy.eq(wait_policy).all() or not raw.total_vehicles.eq(VEHICLES).all():
        raise ValueError("raw.csv 与 metadata 的实验配置不一致。")
    return metadata, summary, raw

if RUN_DIR is None:
    skipped = []
    candidates = sorted(RESULT_ROOT.glob("*/summary.csv"),
                        key=lambda p: p.stat().st_mtime_ns, reverse=True)
    for candidate in candidates:
        try:
            metadata, summary, raw = load_complete_run(candidate.parent)
        except (OSError, ValueError, KeyError, pd.errors.ParserError) as error:
            skipped.append((candidate.parent.name, str(error)))
            continue
        RUN_DIR = candidate.parent
        break
    else:
        raise FileNotFoundError(f"没有找到 {VEHICLES} 车、wait_policy={WAIT_POLICY} 的完整运行。跳过项: {skipped}")
else:
    RUN_DIR = Path(RUN_DIR).expanduser()
    if not RUN_DIR.is_absolute():
        RUN_DIR = PROJECT_ROOT / RUN_DIR
    metadata, summary, raw = load_complete_run(RUN_DIR)

RUN_DIR = RUN_DIR.resolve()
FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
station_metrics = pd.read_csv(RUN_DIR / "station_metrics.csv")
scenarios = [s for s in SCENARIO_ORDER if s in summary.scenario.unique()]
station_counts = [c for c in STATION_COUNTS if c in summary.station_count.unique()]
if not scenarios or not station_counts:
    raise ValueError("没有匹配到所选择的场景或站点数量。")
print(f"Run: {RUN_DIR}\nWait policy: {WAIT_POLICY}\nVehicles: {VEHICLES}")
print(f"Scenarios: {scenarios}; stations: {station_counts}; seeds: {metadata['args']['seeds']}")
missing = set(product(SCENARIO_ORDER, STATION_COUNTS)) - set(zip(summary.scenario, summary.station_count))
if missing:
    print(f"本次运行未包含的场景/规模（不补 0）: {sorted(missing)}")
print("Queue replay:", metadata.get("queue_replay", "未提供"))
print("Figure output:", FIGURE_DIR)

columns = ["scenario", "station_count", "method", "num_seeds", "successful_runs",
           "assigned_charge_vehicles_mean", "idle_capacity_fraction_mean",
           "idle_slots_at_epoch_1_mean", "new_queue_vehicle_count_mean",
           "overcapacity_slot_epochs_mean", "preprocess_seconds_mean"]
display(summary[columns].style.format({c: "{:.3f}" for c in columns if c.endswith("_mean")}, na_rep="NA"))
failed = raw.loc[raw.status != "optimal_on_final_graph", ["scenario", "station_count", "seed", "method", "status"]]
if not failed.empty:
    print("以下求解未成功；均值仅覆盖有指标的成功样本，不能把这些缺失当作 0：")
    display(failed)


## 充电分配量与容量空置

柱为 `summary.csv` 的均值，误差棒为配对种子间的样本标准差（不是置信区间）。只有一个种子时无法估计样本标准差，图中不画误差棒。各方法的成功数 / 总种子数显示在横轴下；失败样本不记为零。

各场景分别代表同时到达、错峰到达、已有充电与确定在途背景、以及含较多非充电偏好的决策。`idle_capacity_fraction` 的绝对水平受该场景共同观察窗口影响，宜首先比较**同一场景内**两种方法。充电车辆数可超过同时提供的 slots，因为车辆可以错峰使用。


In [ ]:
fig, axes = plt.subplots(len(station_counts), 2, figsize=(14, 4.2 * len(station_counts)),
                         squeeze=False, constrained_layout=True)
metrics = [("assigned_charge_vehicles", "Assigned charging vehicles", False),
           ("idle_capacity_fraction", "Idle capacity", True)]
width = 0.34
for row_index, count in enumerate(station_counts):
    present_scenarios = [s for s in scenarios
                         if ((summary.scenario == s) & (summary.station_count == count)).any()]
    x = np.arange(len(present_scenarios))
    group = summary.loc[summary.station_count == count].set_index(["scenario", "method"])
    tick_labels = []
    for scenario in present_scenarios:
        rates = []
        for method in METHODS:
            if (scenario, method) in group.index:
                record = group.loc[scenario, method]
                rates.append(f"{int(record.successful_runs)}/{int(record.num_seeds)}")
            else:
                rates.append("NA")
        tick_labels.append(f"{SCENARIO_LABELS[scenario]}\n" + " / ".join(rates))
    for col_index, (metric, label, percentage) in enumerate(metrics):
        ax = axes[row_index, col_index]
        for method_index, method in enumerate(METHODS):
            offset = (method_index - 0.5) * width
            for position, scenario in enumerate(present_scenarios):
                if (scenario, method) not in group.index:
                    continue
                record = group.loc[scenario, method]
                mean = record[f"{metric}_mean"]
                if not np.isfinite(mean):
                    ax.text(position + offset, 0.02, "NA", ha="center",
                            transform=ax.get_xaxis_transform(), color=COLORS[method])
                    continue
                std = record[f"{metric}_std"]
                error = std if int(record.successful_runs) > 1 and np.isfinite(std) else None
                ax.bar(position + offset, mean, width, color=COLORS[method],
                       yerr=error, capsize=3, error_kw={"linewidth": 1.1})
        ax.set_xticks(x, tick_labels)
        ax.set_title(f"{count} stations × {metadata['args']['slots']} slots")
        ax.set_ylabel(label)
        ax.grid(axis="y", color="#dce1e6", linewidth=0.7)
        if percentage:
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            ax.set_ylim(0, 1.05)
        else:
            ax.set_ylim(bottom=0)
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.margins(y=0.12)
        ax.set_xlabel("Success / seeds: Current / Conservative")
        ax.legend(handles=[Patch(facecolor=COLORS[m], label=METHOD_LABELS[m]) for m in METHODS],
                  loc="upper right", frameon=True, fontsize=9)
fig.suptitle(f"Charging admission with {VEHICLES} vehicles · wait policy: {WAIT_POLICY}", fontsize=15)
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_DIR / f"charging_comparison_{WAIT_POLICY}.{suffix}", bbox_inches="tight")
plt.show()


## 单个种子的站点占用日历

在下一格选择场景、站点数和种子；`None` 自动选择一个同时具有两种方法占用数据的 case。灰色为已充电 / 已确定到达的背景预约，彩色为本轮实际选中的新充电车辆，虚线为实际 slots 容量。横轴是 **模拟 epoch，不是分钟**，每个阶梯表示半开区间 `[epoch, epoch + 1)` 的占用。黑色点线为实际固定日历回放占用，可检查它是否与预约相符。


In [ ]:
CASE_SCENARIO = None  # 例如 "synchronized"、"committed"
CASE_STATIONS = None  # 3 或 5
CASE_SEED = None      # 例如 0

available_cases = []
for scenario, count, seed in raw[["scenario", "station_count", "seed"]].drop_duplicates().itertuples(index=False, name=None):
    case_dir = RUN_DIR / "cases" / f"{scenario}_{count}stations_seed{seed}"
    if all((case_dir / f"{method}_occupancy.npz").is_file() for method in METHODS):
        available_cases.append((scenario, int(count), int(seed)))
available_cases.sort(key=lambda item: (SCENARIO_ORDER.index(item[0]), item[1], item[2]))
selected = next((item for item in available_cases
                 if (CASE_SCENARIO is None or item[0] == CASE_SCENARIO)
                 and (CASE_STATIONS is None or item[1] == CASE_STATIONS)
                 and (CASE_SEED is None or item[2] == CASE_SEED)), None)
if selected is None:
    raise ValueError(f"所选 case 没有两种方法的完整 occupancy 数据。可选项：{available_cases}")
scenario, count, seed = selected
case_dir = RUN_DIR / "cases" / f"{scenario}_{count}stations_seed{seed}"
info = json.loads((case_dir / "input.json").read_text())
profiles = {}
for method in METHODS:
    with np.load(case_dir / f"{method}_occupancy.npz") as archive:
        profiles[method] = {key: archive[key].copy() for key in archive.files}
station_ids = sorted(int(key.split("_")[1]) for key in profiles[METHODS[0]] if key.endswith("_base"))
print(f"Selected: {scenario}, {count} stations, seed {seed}; {info['total_vehicles']} total vehicles")
case_metrics = station_metrics.loc[(station_metrics.scenario == scenario)
                                  & (station_metrics.station_count == count)
                                  & (station_metrics.seed == seed)]
display(case_metrics[["station_id", "method", "assigned_charge_vehicles", "idle_slot_epochs",
                      "idle_slots_at_epoch_1", "new_queue_vehicle_count", "new_queue_wait_epochs",
                      "overcapacity_slot_epochs"]].sort_values(["station_id", "method"]))


In [ ]:
fig, axes = plt.subplots(len(station_ids), 2, figsize=(13, 2.5 * len(station_ids)),
                         squeeze=False, sharex=True, sharey=True, constrained_layout=True)
peak = max(float((p[f"station_{sid}_base"] + p[f"station_{sid}_new"]).max())
           for p in profiles.values() for sid in station_ids)
capacity = info["slots_per_station"]
for row_index, sid in enumerate(station_ids):
    for col_index, method in enumerate(METHODS):
        ax = axes[row_index, col_index]
        base = profiles[method][f"station_{sid}_base"]
        new = profiles[method][f"station_{sid}_new"]
        replay = profiles[method][f"station_{sid}_replay"]
        if not (len(base) == len(new) == len(replay)):
            raise ValueError(f"占用数组长度不一致: {method}, station {sid}")
        x = np.arange(len(base) + 1)
        base_step, total_step = np.r_[base, base[-1]], np.r_[base + new, (base + new)[-1]]
        ax.fill_between(x, 0, base_step, step="post", color="#bfc7d1")
        ax.fill_between(x, base_step, total_step, step="post", color=COLORS[method], alpha=0.85)
        ax.step(x, np.r_[replay, replay[-1]], where="post", color="#333333", linestyle=":", linewidth=1.2)
        ax.axhline(capacity, color="#C64F53", linestyle="--", linewidth=1.4)
        ax.set_title(f"Station {sid} · {METHOD_LABELS[method]}")
        ax.set_ylim(0, max(capacity, peak) * 1.15)
        ax.set_xlim(0, len(base))
        ax.grid(axis="y", color="#e5e8eb", linewidth=0.6)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
        ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
        if col_index == 0:
            ax.set_ylabel("Occupied slots")
        if row_index == len(station_ids) - 1:
            ax.set_xlabel("Epoch since decision (not minutes)")
fig.legend(handles=[Patch(facecolor="#bfc7d1", label="Committed background"),
                    Patch(facecolor=COLORS["current"], label="Current new assignments"),
                    Patch(facecolor=COLORS["conservative"], label="Conservative new assignments"),
                    Line2D([0], [0], color="#333333", linestyle=":", label="Fixed-calendar replay"),
                    Line2D([0], [0], color="#C64F53", linestyle="--", label=f"Capacity ({capacity})")],
           loc="outside lower center", fontsize=9, ncol=3, frameon=False)
fig.suptitle(f"{SCENARIO_LABELS[scenario]} · {count} stations · seed {seed} · wait policy: {WAIT_POLICY}", fontsize=14)
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_DIR / f"occupancy_{scenario}_{count}stations_seed{seed}_{WAIT_POLICY}.{suffix}", bbox_inches="tight")
plt.show()


`new_queue_vehicle_count = 0` 表明**本次固定日历回放的新车辆**没有等待；已有车辆被提前排入背景预约时产生的延迟另见 `background_scheduled_wait_epochs`。不能用总空位或当前时点的空位来代替完整服务窗口可行性，也不能用单轮回放直接断言全天 NYC 仿真没有堵塞。

SSG 的对照值 `ssg_on_off_objective_gap_int` 应在同一方法、同一固定可行图下检查。conservative 改变原问题后，分数与 current 不同本身不说明 SSG 失去该固定图上的精确性。
